# 01 — Artifact inventory (all 519 reference genomes)

**Goal:** census every KBase artifact tied to the Watershed phenotype-replication
study across the three involved workspaces -- narrative `119455`, models `220871`,
and `KBaseMedia` -- and report, per reference genome, which artifact classes
(genome, old model, new model, observed phenotype set, prior simulation set,
media, assembly) are present or missing.

**Provenance correction (see also the markdown cell + TSV header below):** an
earlier pass concluded "only 3/519 genomes have an old model" by listing
`KBaseFBA.FBAModel` objects in `MODELS_WS` (220871) only -- the workspace that
holds the *new* genomeset-rebuilt models. The "old" models were built directly
against the Hope College genome narrative and live in `NARRATIVE_WS` (119455).
This notebook censuses **all** involved workspaces and splits old/new by source
workspace (`census_artifacts` / `partition_models`), so that undercount cannot
recur silently.

**Outputs** (written to `NBOutput/`):
- `artifact_inventory.tsv` -- one row per censused artifact (see
  `util.census_artifacts` for the exact column list), with the provenance note
  above as a literal comment line at the top of the file.
- `artifact_missing.tsv` -- one row per reference genome, listing which
  artifact classes it is missing.

Each cell starts with `%run util.py` (no per-cell imports) and is independent,
loading its inputs from `session.cache`. This notebook is authored and
statically validated only -- it is **not** executed against live KBase as part
of this task.

## Step 1 — Load reference genomes and run the census

`load_watershed_sheet()` gives the 519 reference genome ids from the
collaborators' mastersheet (the study's canonical genome list). `census_artifacts`
walks `NARRATIVE_WS`, `MODELS_WS`, and `KBASEMEDIA_WS` via
`MSFBAUtils.list_ws_objects`, classifies every object by KBase type into an
artifact class, and normalizes each to a reference genome id (metadata
`genome_ref` preferred, else a reference-token extracted from the object name;
ambiguous/unresolvable objects are excluded and reported rather than guessed).

In [ ]:
%run util.py

ensure_kbase_token()
fba = get_msfba()

ref_genomes, media_names, watershed_df = load_watershed_sheet()
print(f"{len(ref_genomes)} reference genomes loaded from {WATERSHED_CSV.name}")

workspaces = [NARRATIVE_WS, MODELS_WS, KBASEMEDIA_WS]
table_df, missing_report = census_artifacts(workspaces, ref_genomes=ref_genomes, fba=fba)

print(f"{len(table_df)} censused artifacts across {len(workspaces)} workspaces")
print(table_df["artifact_class"].value_counts())

session.cache.save("artifact_table", table_df)
session.cache.save("artifact_missing_report", missing_report)

## Step 2 — Old-model count: corrected vs. prior (wrong-workspace) conclusion

Sanity-checks the provenance correction directly: counts genomes with at least
one `old_model` row (source = `NARRATIVE_WS`, 119455) against the prior "3/519"
conclusion, which came from scanning `MODELS_WS` (220871) for old models --
a workspace that does not contain them.

In [ ]:
%run util.py

table_df = session.cache.load("artifact_table")
ref_genomes, media_names, watershed_df = load_watershed_sheet()

old_model_genomes = set(table_df.loc[table_df["artifact_class"] == "old_model", "genome_id"])
new_model_genomes = set(table_df.loc[table_df["artifact_class"] == "new_model", "genome_id"])

print(f"old_model present for {len(old_model_genomes)}/{len(ref_genomes)} reference genomes "
      f"(source: NARRATIVE_WS {NARRATIVE_WS})")
print(f"new_model present for {len(new_model_genomes)}/{len(ref_genomes)} reference genomes "
      f"(source: MODELS_WS {MODELS_WS})")
print()
print("Prior (wrong-workspace) conclusion: only 3/519 genomes have an old model "
      "-- that count came from scanning MODELS_WS for KBaseFBA.FBAModel, which "
      "only holds the *new* genomeset-rebuilt models. The corrected old_model "
      "count above comes from NARRATIVE_WS, the actual source workspace.")

## Step 3 — Write `artifact_inventory.tsv`

The provenance-correction note (`util.PROVENANCE_NOTE`) is written as a literal
`#`-prefixed comment line at the top of the file, ahead of the TSV header row,
so the correction travels with the data.

In [ ]:
%run util.py

table_df = session.cache.load("artifact_table")

NBOUTPUT_DIR.mkdir(parents=True, exist_ok=True)
inventory_path = NBOUTPUT_DIR / "artifact_inventory.tsv"
with open(inventory_path, "w") as fh:
    fh.write(f"# {PROVENANCE_NOTE}\n")
    table_df.to_csv(fh, sep="\t", index=False)

print("wrote", inventory_path, f"({len(table_df)} rows)")

## Step 4 — Missing-artifact report

Flattens `missing_report["per_genome"]` into a table (one row per reference
genome, listing which artifact classes it lacks) and prints the roll-up counts
(`missing_report["counts"]` / `missing_report["missing_counts"]`).

In [ ]:
%run util.py

missing_report = session.cache.load("artifact_missing_report")

missing_rows = []
for gid, missing_classes in missing_report["per_genome"].items():
    missing_rows.append({
        "genome_id": gid,
        "n_missing": len(missing_classes),
        "missing_classes": ",".join(missing_classes),
    })
missing_df = pd.DataFrame(missing_rows).sort_values(
    ["n_missing", "genome_id"], ascending=[False, True]
).reset_index(drop=True)

print(f"{(missing_df['n_missing'] > 0).sum()}/{len(missing_df)} reference genomes "
      "are missing at least one artifact class")
print()
print("roll-up counts (total artifacts found per class):")
print(missing_report["counts"])
print()
print("roll-up counts (genomes missing each class):")
print(missing_report["missing_counts"])
print()
print(f"{len(missing_report['unmappable'])} objects could not be normalized to a "
      "reference genome (see missing_report['unmappable'])")

missing_path = NBOUTPUT_DIR / "artifact_missing.tsv"
missing_df.to_csv(missing_path, sep="\t", index=False)
print("wrote", missing_path, f"({len(missing_df)} rows)")

## Summary

- `NBOutput/artifact_inventory.tsv` -- full per-artifact census, with the
  provenance-correction note as its first line.
- `NBOutput/artifact_missing.tsv` -- per-genome missing-artifact-class report.
- The old/new model split is by source workspace (`NARRATIVE_WS` /
  `MODELS_WS`), not by name heuristic, and the corrected old-model count
  supersedes the prior "3/519" (wrong-workspace) conclusion.

Next: use `artifact_missing.tsv` to scope the gapfilling/backfill work needed
before the phenotype-replication sweep can run over the full 519-genome set.